In [1]:
import torch
import torch.nn as nn
# 参考: https://blog.csdn.net/Mr_health/article/details/122822483

<span style='color:red;'>1.单机多卡</span>

In [ ]:
# b. 方式二:(适用与多节点多卡和单机多卡)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
# 1). 初始化
from torch.utils.data.distributed import DistributedSampler
# torch.distributed.init_process_group(backend="nccl")
# 初始化分布式进程组，并指定每个进程的等级
torch.distributed.init_process_group(backend='nccl', init_method='env://', world_size=3, rank=0)

input_size = 5
output_size = 2
batch_size = 30
data_size = 90

# 2). 配置每一个进程的gpu
local_rank = torch.distributed.get_rank() # 获取当前进程的等级
print('local_rank',local_rank)
torch.cuda.set_device(local_rank)
device = torch.device("cuda", local_rank)

class RandomDataset(Dataset):
    def __init__(self, size, length):
        self.len = length
        self.data = torch.randn(length, size).to('cuda')
 
    def __getitem__(self, index):
        return self.data[index]
 
    def __len__(self):
        return self.len
 
dataset = RandomDataset(input_size, data_size)

# 3）使用DistributedSampler ---------------- 区别于单卡的地方
rand_loader = DataLoader(dataset=dataset,
                         batch_size=batch_size,
                         sampler=DistributedSampler(dataset))

class Model(nn.Module):
    def __init__(self, input_size, output_size):
        super(Model, self).__init__()
        self.fc = nn.Linear(input_size, output_size)
 
    def forward(self, input):
        output = self.fc(input)
        print("  In Model: input size", input.size(),
              "output size", output.size())
        return output

model = Model(input_size, output_size)

# 4) 封装之前要把模型移到对应的gpu
model.to(device)


if torch.cuda.device_count() > 1:
    print("Let's use", torch.cuda.device_count(), "GPUs!")
    # 5) 封装
    model = torch.nn.parallel.DistributedDataParallel(model,
                                                      device_ids=[local_rank],
                                                      output_device=local_rank)
    

for data in rand_loader:
    if torch.cuda.is_available():
        input_var = data
    else:
        input_var = data
 
    output = model(input_var)
    print("Outside: input size", input_var.size(), "output_size", output.size())

    
    """
    （1）启动方式：在torch.distributed当中提供了一个用于启动的程序torch.distributed.launch，
        此帮助程序可用于为每个节点启动多个进程以进行分布式训练，
        它在每个训练节点上产生多个分布式训练进程。
    （2）启动命令：
        CUDA_VISIBLE_DEVICES=1,2,3,4 python -m torch.distributed.launch --nproc_per_node=2 torch_ddp.py
    这里需要说明一下参数：
        CUDA_VISIBLE_DEVICES：设置我们可用的GPU的id
        torch.distributed.launch：用于启动多节点多GPU的训练
        nproc_per_node：表示设置的进程数量，一般情况设置为可用的GPU数量，即有多少个可用的GPU就设置多少个进程。
        local rank：关于这个参数的意义，我们将在后面的情形中进行说明。
             local rank：表示的是当前的进程在当前节点的编号，因为我们设置了2个进程(因为nproc_per_node=2)，因此进程的编号就是0和1
             在很多博客中都直接说明local_rank等于进程内的GPU编号，这种说法实际上是不准确的。这个编号并不是GPU的编号！！
    
    
    
    
    """
    
    
    
    













<span style='color:red;'>2.注意事项<span>

In [17]:
import numpy as np
def get_box_area(box):
    
    return (box[2]-box[0]) * (box[3]-box[1])

def cal_iou(box_a, box_b):
    print(box_a[:2])
    left = max(box_a[0],box_b[0])
    top = max(box_a[1],box_b[1])
    right = min(box_a[2],box_b[2])
    button = max(box_a[3],box_b[3])
    
    iou_area = get_box_area([left,top,right,button])
   
    box_a_area = get_box_area(box_a)
    box_b_area = get_box_area(box_b)
    print(iou_area,box_a_area,box_b_area)
    return iou_area/(box_a_area + box_b_area - iou_area)*1.0

cal_iou(np.array([0,0,100,100]),np.array([50,50,100,100]))

[0 0]
2500 10000 2500


0.25

In [4]:
def nms(boxes, iou_threshold):
    """
    boxes=[[x0,y0,x1,y1,score],...]
    """
    boxes = sorted(boxes,key=lambda x:x[-1])[::-1]
    best_score_box = boxes[0]
    reserv_boxes = []
    reserv_boxes.append(best_score_box)
    for box in boxes[1:]:
        if cal_iou(best_score_box,box) < iou_threshold:
            reserv_boxes.append(box)
    return reserv_boxes
    
    
    
    
    
    

[1, 2, 3, 4]

In [19]:
sorted([1,2,4,3])[::-1]

[4, 3, 2, 1]

In [11]:
a = [1,2,3,4]
a[:2]
b = np.array([0,0,100,100])
b[:2]

array([0, 0])

In [25]:
a = [[1,2,4]]
a.append([2,3,4])
print(a)

[[1, 2, 4], [2, 3, 4]]


In [26]:
    import numpy as np
    x = np.random.randn(6, 6)
    print(x)
    print('*'*10)
    x_min = np.min(x)
    print(x_min)

[[ 0.5987573   0.44540324  0.86643132 -0.20798468  0.96991787  0.69109387]
 [ 1.1362793   1.50047301 -1.13229254 -0.56397779 -0.05194989 -3.0432507 ]
 [-0.05455347 -1.85594069  1.79296235  1.94063797  2.04854714 -1.96089893]
 [ 1.57471587  1.92181917  0.87216744  0.06548274 -0.64308566  0.97984852]
 [-0.37971697  0.27636478 -1.30156857  1.60274234  0.19213153 -0.51813694]
 [ 1.27368417  0.03036366  0.8041467  -1.57285513  0.96706301 -0.93359355]]
**********
-3.043250700313771


In [43]:
list1 = [
    [0, 1, 1, 1, 0],
    [0, 1, 0, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0]
]
image = np.asarray(list1)
image

array([[0, 1, 1, 1, 0],
       [0, 1, 0, 1, 0],
       [0, 1, 1, 1, 0],
       [0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0]])

In [46]:
image = np.asarray(list1)
np.roll(image,shift=(1,0),axis=(0,1))

array([[0, 0, 0, 1, 0],
       [0, 1, 1, 1, 0],
       [0, 1, 0, 1, 0],
       [0, 1, 1, 1, 0],
       [0, 0, 0, 1, 0]])

In [32]:
image[...,0]

array([0, 0, 0, 0, 0])

In [31]:
image

array([[0, 1, 1, 1, 0],
       [0, 1, 0, 1, 0],
       [0, 1, 1, 1, 0],
       [0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0]])